In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import time

# 1. 데이터 로드 및 전처리
print("MNIST 데이터 로드 및 전처리 중...")
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)

# 정규화 (0-255 -> 0.0-1.0)
X = X / 255.0
y = y.astype(int)

# X의 형태를 확인합니다. (70000, 784) 형태여야 합니다.
print(f"데이터 로드 완료. X.shape: {X.shape}")


MNIST 데이터 로드 및 전처리 중...
데이터 로드 완료. X.shape: (70000, 784)


In [ ]:
# 2. 탐색할 하이퍼파라미터 그리드 정의
param_grid_knn = {
    'n_neighbors': [3, 5, 7], # 이웃 수 (K)
    'weights': ['uniform', 'distance'], # 가중치 방식
    'p': [1, 2] # 거리 방식 (L1: 맨하튼, L2: 유클리드)
}

# 3. 교차 검증 객체 설정 (CV=3 사용)
cv_knn = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [ ]:

# 4. GridSearchCV 설정 및 실행
knn_clf = KNeighborsClassifier(n_jobs=-1)

grid_search_knn = GridSearchCV(
    estimator=knn_clf,
    param_grid=param_grid_knn,
    cv=cv_knn,
    scoring='accuracy',
    verbose=2, # 진행 상황 상세 출력
    n_jobs=-1 # 탐색 과정 전체 병렬 처리 (CPU 코어 전체 사용)
)

print("\n--- Grid Search for KNN 시작 (시간 소요 예상) ---")
start_time = time.time()
grid_search_knn.fit(X, y)
end_time = time.time()

# 5. 최적 결과 출력
print("--- 최적 파라미터 탐색 완료 ---")
print(f"총 소요 시간: {end_time - start_time:.2f} 초")
print(f"최적의 파라미터 조합: {grid_search_knn.best_params_}")
print(f"최고 교차 검증 정확도: {grid_search_knn.best_score_:.4f}")


--- Grid Search for KNN 시작 (시간 소요 예상) ---
Fitting 5 folds for each of 12 candidates, totalling 60 fits
--- 최적 파라미터 탐색 완료 ---
총 소요 시간: 2841.73 초
최적의 파라미터 조합: {'n_neighbors': 3, 'p': 2, 'weights': 'distance'}
최고 교차 검증 정확도: 0.9738


In [ ]:
import pandas as pd

cvres = (pd.DataFrame(grid_search_knn.cv_results_)
           .sort_values("mean_test_score", ascending=False)
           .head(10))
cvres

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_n_neighbors,param_p,param_weights,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
3,2.420986,3.114043,90.605427,10.034743,3,2,distance,"{'n_neighbors': 3, 'p': 2, 'weights': 'distance'}",0.975286,0.973000,0.972000,0.974929,0.973714,0.973786,0.001214,1
2,8.029514,0.543222,90.884079,6.000573,3,2,uniform,"{'n_neighbors': 3, 'p': 2, 'weights': 'uniform'}",0.973857,0.971286,0.971214,0.974500,0.973000,0.972771,0.001330,2
7,1.249546,0.077247,139.550822,5.416037,5,2,distance,"{'n_neighbors': 5, 'p': 2, 'weights': 'distance'}",0.972643,0.973143,0.971000,0.973214,0.972286,0.972457,0.000804,3
11,1.239232,0.058370,141.495521,6.607442,7,2,distance,"{'n_neighbors': 7, 'p': 2, 'weights': 'distance'}",0.970929,0.971857,0.969714,0.972500,0.971786,0.971357,0.000962,4
6,1.285177,0.058641,139.701518,9.383694,5,2,uniform,"{'n_neighbors': 5, 'p': 2, 'weights': 'uniform'}",0.970857,0.971857,0.969786,0.972214,0.971143,0.971171,0.000846,5
10,1.308659,0.056784,145.202818,10.594505,7,2,uniform,"{'n_neighbors': 7, 'p': 2, 'weights': 'uniform'}",0.969786,0.970500,0.969357,0.971786,0.970786,0.970443,0.000841,6
1,7.873527,0.332898,1241.475063,63.668829,3,1,distance,"{'n_neighbors': 3, 'p': 1, 'weights': 'distance'}",0.969714,0.967857,0.965357,0.968643,0.967429,0.967800,0.001448,7
5,1.275029,0.229026,1339.539993,141.440375,5,1,distance,"{'n_neighbors': 5, 'p': 1, 'weights': 'distance'}",0.966071,0.967714,0.965286,0.967786,0.966357,0.966643,0.000970,8
0,7.766039,0.420072,1214.768144,145.415801,3,1,uniform,"{'n_neighbors': 3, 'p': 1, 'weights': 'uniform'}",0.967714,0.966286,0.964286,0.967786,0.966071,0.966429,0.001283,9
9,1.280730,0.056710,1296.725805,32.591977,7,1,distance,"{'n_neighbors': 7, 'p': 1, 'weights': 'distance'}",0.965214,0.965786,0.963643,0.967071,0.965286,0.965400,0.001103,10


In [ ]:

optimal_params = {
    'n_neighbors': 3,
    'weights': 'distance',
    'p': 2
}
final_knn_model = KNeighborsClassifier(
    **optimal_params,
    n_jobs=-1
)


In [ ]:
# 4. 전체 7만 장 데이터로 최종 훈련 (Fit)
# KNN은 fit 단계는 빠르지만, 모델 객체 생성 및 검증 단계에서 메모리를 많이 사용합니다.
print("--- 최종 KNN 모델 훈련 시작 (빠름) ---")
start_time = time.time()
final_knn_model.fit(X, y)
end_time = time.time()

print(f"--- 최종 모델 훈련 완료 ---")
print(f"훈련 소요 시간: {end_time - start_time:.2f} 초")

# 최종 모델을 'best_model' 변수에 할당하여 joblib 저장 준비
best_model = final_knn_model

--- 최종 KNN 모델 훈련 시작 (빠름) ---
--- 최종 모델 훈련 완료 ---
훈련 소요 시간: 0.62 초


In [ ]:
import joblib
import os

MODEL_FILENAME = "knn_final_model.joblib"

try:
    joblib.dump(best_model, MODEL_FILENAME)

    print("---")
    print(f"✅ 최종 모델 저장 성공!")
    print(f"파일 경로: {os.path.abspath(MODEL_FILENAME)}")
    print("---")

except Exception as e:
    print(f"❌ 모델 저장 실패: {e}")

---
✅ 최종 모델 저장 성공!
파일 경로: C:\mechinelearning\mnist_image_trasform\knn_final_model.joblib
---
